# Feature Engineering

For this analysis, we have transformed raw price data into advanced financial metrics. These variables allow us to capture XRP market dynamics in a way that raw prices cannot.

---

### 1. Logarithmic Returns (`Log_Returns`)
* **Definition:** The natural logarithm of the ratio between the current closing price and the previous closing price: $r_t = \ln(P_t / P_{t-1})$.
* **Utility:** In quantitative finance, these are preferred over simple returns because they are **time-additive** (you can sum daily returns to get monthly ones) and help normalize the series, facilitating the use of statistical models.

### 2. Realized Volatility (`Volatility_30d`)
* **Definition:** The rolling standard deviation of `Log_Returns` calculated over a 30-day window.
* **Utility:** It measures the **risk or uncertainty** of the asset in the short term. High volatility indicates large price swings (fear or euphoria), while low volatility indicates stability. It is key for detecting "panic" periods.

### 3. Intraday Range Percentage (`Intraday_Range_Pct`)
* **Definition:** The percentage difference between the day's maximum and minimum: $((High - Low) / Low) * 100$.
* **Utility:** It captures **within-day nervousness**. It helps identify "exhaustion" candles or days where trading was particularly aggressive regardless of the final close.

In [1]:
import pandas as pd
import numpy as np
import os

# 1. Load the data
input_path = 'items/XRP_Data.csv'
output_path = 'items/XRP_Features.csv'

df = pd.read_csv(input_path)
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').set_index('Date')

# --- 2. CALCULATING FINANCIAL FEATURES ---

# A. Logarithmic Returns
df['Log_Returns'] = np.log(df['Close'] / df['Close'].shift(1))

# B. Realized Volatility (Rolling 30-day Std Dev)
# We multiply by sqrt(365) to annualize it if needed, but here we keep it daily
df['Volatility_30d'] = df['Log_Returns'].rolling(window=30).std()

# C. Intraday Range (Percentage)
df['Intraday_Range_Pct'] = ((df['High'] - df['Low']) / df['Low']) * 100

# D. Target Variable (Next Day Direction - for future ML models)
df['Target_Next_Day'] = np.where(df['Log_Returns'].shift(-1) > 0, 1, 0)

print("✅ Features calculated successfully.")

✅ Features calculated successfully.


## Step 2: Data Cleaning and Truncation

When calculating rolling windows (like 30-day volatility), the first 29 records will result in `NaN` (Not a Number). To maintain high analytical standards, we must remove these rows before proceeding with the statistical study.

In [2]:
# --- 3. CLEANING ---

# Remove rows with null values resulting from rolling calculations
df_clean = df.dropna().copy()

# --- 4. EXPORT AND VERIFICATION ---

# Save the processed dataset
df_clean.to_csv(output_path, index=True)

print(f"✅ Process completed successfully.")
print(f"📊 Original records: {len(df)}")
print(f"🧹 Records after cleaning: {len(df_clean)}")
print(f"💾 File saved at: {output_path}")

# Quick visualization of the new features
display(df_clean[['Close', 'Log_Returns', 'Volatility_30d', 'Intraday_Range_Pct']].head())

✅ Process completed successfully.
📊 Original records: 3044
🧹 Records after cleaning: 3014
💾 File saved at: items/XRP_Features.csv


,Close,Log_Returns,Volatility_30d,Intraday_Range_Pct
Date,,,,
2017-12-09,0.244708,-0.029859,0.061812,6.502603
2017-12-10,0.237333,-0.030601,0.061268,7.084043
2017-12-11,0.251691,0.058738,0.062015,7.197928
2017-12-12,0.373541,0.394826,0.092994,72.975876
2017-12-13,0.471063,0.231964,0.100643,49.802319


An initial truncation of the dataset was performed to eliminate bias produced by the convergence period of the moving indicators. This ensures that subsequent statistical analysis is based exclusively on observations with fully calculated return and volatility metrics, avoiding the introduction of noise through data imputation.

